In [1]:
#Install dependencies
!pip install great-expectations==0.18.21 numpy==1.26.4 pandas==2.2.2 pytest -q

In [2]:
# Generate the messy customer_data.csv
import pandas as pd
import numpy as np
import os

np.random.seed(42)
n = 700

# --- Clean base data ---
data = {
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n + 1)],
    "age": np.random.randint(18, 80, n).tolist(),
    "email": [f"user{i}@example.com" for i in range(1, n + 1)],
    "salary": np.random.uniform(30000, 120000, n).tolist(),
    "country": np.random.choice(["USA", "Canada", "UK", "Australia"], n).tolist(),
    "phone": [f"416-555-{str(i).zfill(4)}" for i in range(1, n + 1)],
    "signup_date": pd.date_range("2020-01-01", periods=n, freq="D").astype(str).tolist(),
}

df = pd.DataFrame(data)

# --- Inject data quality issues ---

# 1. Missing values in age (10 rows)
df.loc[np.random.choice(n, 10, replace=False), "age"] = np.nan

# 2. Missing values in email (15 rows)
df.loc[np.random.choice(n, 15, replace=False), "email"] = np.nan

# 3. Missing values in salary (40 rows) — must stay above 95% present
df.loc[np.random.choice(n, 40, replace=False), "salary"] = np.nan

# 4. Duplicate customer records (5 duplicates)
duplicates = df.iloc[:5].copy()
df = pd.concat([df, duplicates], ignore_index=True)

# 5. Out-of-range age values
df.loc[0, "age"] = 150
df.loc[1, "age"] = -5

# 6. Negative salary
df.loc[2, "salary"] = -5000

# 7. Salary stored as string with dollar signs (20 rows)
str_indices = np.random.choice(n, 20, replace=False)
for idx in str_indices:
    val = df.loc[idx, "salary"]
    if pd.notna(val):
        df.loc[idx, "salary"] = f"${val:,.2f}"

# 8. Invalid email formats
df.loc[3, "email"] = "not-an-email"
df.loc[4, "email"] = "missing@"
df.loc[5, "email"] = "@nodomain.com"

# 9. Invalid country
df.loc[6, "country"] = "Germany"
df.loc[7, "country"] = "France"

# 10. Inconsistent phone formats
df.loc[10, "phone"] = "(416) 555-0010"
df.loc[11, "phone"] = "4165550011"
df.loc[12, "phone"] = "+1-416-555-0012"

# --- Save ---
os.makedirs("data", exist_ok=True)
df.to_csv("data/customer_data.csv", index=False)

print(f"✅ Dataset created: {len(df)} rows, {len(df.columns)} columns")
print(df.head())

✅ Dataset created: 705 rows, 7 columns
  customer_id    age              email         salary country         phone  \
0       C0001  150.0  user1@example.com  103838.415939     USA  416-555-0001   
1       C0002   -5.0  user2@example.com            NaN  Canada  416-555-0002   
2       C0003   46.0  user3@example.com        -5000.0     USA  416-555-0003   
3       C0004   32.0       not-an-email   54656.501459      UK  416-555-0004   
4       C0005   60.0           missing@   49313.020774     USA  416-555-0005   

  signup_date  
0  2020-01-01  
1  2020-01-02  
2  2020-01-03  
3  2020-01-04  
4  2020-01-05  


/tmp/ipykernel_4022/1321008132.py:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '$96,262.25' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, "salary"] = f"${val:,.2f}"


In [3]:
# Verify the issues are there
df_check = pd.read_csv("data/customer_data.csv")

print("Shape:", df_check.shape)
print("\nMissing values:")
print(df_check.isnull().sum())
print("\nDuplicate rows:", df_check.duplicated().sum())
print("\nSample of salary column (mixed types):")
print(df_check["salary"].head(25))

Shape: (705, 7)

Missing values:
customer_id     0
age            10
email          15
salary         41
country         0
phone           0
signup_date     0
dtype: int64

Duplicate rows: 0

Sample of salary column (mixed types):
0     103838.41593941413
1                    NaN
2                -5000.0
3      54656.50145890204
4      49313.02077375367
5     63953.317490517824
6     33506.681907921025
7      85642.84959041036
8      60289.87933998539
9       89015.0371842668
10     64685.69588690085
11     91345.23020755107
12      60656.2112517816
13     53462.50754123128
14     74643.37088640656
15     92360.13228227454
16     61350.29440078777
17    114298.33366701365
18     33526.76940080816
19     67615.14285440209
20    117082.24963682228
21     79317.46949232786
22     68112.38480773068
23     81166.82553749625
24     81833.21177322761
Name: salary, dtype: object


In [ ]:
import great_expectations as gx

# Create an in-memory DataContext
context = gx.get_context(project_dir=None)

print("Great Expectations DataContext initialized!")